# Population Estimation: Lincoln-Petersen Index

**Formula:** $\hat{N} = \frac{N_1 \times N_2}{N_b}$

- $N_1$ = number of unique individuals captured on Day 1
- $N_2$ = number of unique individuals captured on Day 2
- $N_b$ = number of individuals captured on **both** days

Each cluster in `lca_annots.json` represents one individual animal.

In [8]:
import json
from datetime import datetime, date
from collections import defaultdict
import math

ANNOTS_PATH = '/fs/ess/PAS2136/ggr_data/results/GGR2024_fixed_encounter/lca/lca_annots.json'

with open(ANNOTS_PATH) as f:
    data = json.load(f)

images = {img['uuid']: img for img in data['images']}
annots = data['annotations']

print(f'Total annotations: {len(annots)}')
print(f'Total images: {len(images)}')

Total annotations: 8006
Total images: 7226


## 1. Identify census days and assign individuals to days

In [9]:
# Map each annotation to its date
day_clusters = defaultdict(set)  # date -> set of cluster_ids

for a in annots:
    img = images[a['image_uuid']]
    dt = datetime.fromtimestamp(img['timestamp'])
    day_clusters[dt.date()].add(a['cluster_id'])

print('Day breakdown:')
print(f'{"Date":<15} {"Day":<12} {"Individuals":>12}')
print('-' * 40)
for d in sorted(day_clusters.keys()):
    print(f'{str(d):<15} {d.strftime("%A"):<12} {len(day_clusters[d]):>12}')

Day breakdown:
Date            Day           Individuals
----------------------------------------
1969-12-31      Wednesday               7
2023-01-27      Friday                  1
2024-01-21      Sunday                 37
2024-01-22      Monday                 32
2024-01-27      Saturday             1139
2024-01-28      Sunday               1070


## 2. Lincoln-Petersen estimate using the two main census days

In [10]:
# The two main GGR census days
DAY1 = date(2024, 1, 27)
DAY2 = date(2024, 1, 28)

N1_set = day_clusters[DAY1]  # individuals seen on Day 1
N2_set = day_clusters[DAY2]  # individuals seen on Day 2
Nb_set = N1_set & N2_set     # individuals seen on both days

N1 = len(N1_set)
N2 = len(N2_set)
Nb = len(Nb_set)

print(f'Day 1 ({DAY1}): N1 = {N1} unique individuals')
print(f'Day 2 ({DAY2}): N2 = {N2} unique individuals')
print(f'Both days:      Nb = {Nb} recaptured individuals')
print()

Day 1 (2024-01-27): N1 = 1139 unique individuals
Day 2 (2024-01-28): N2 = 1070 unique individuals
Both days:      Nb = 554 recaptured individuals



In [11]:
# Lincoln-Petersen estimate
N_hat = (N1 * N2) / Nb
print(f'Lincoln-Petersen estimate: N = N1 * N2 / Nb = {N1} * {N2} / {Nb} = {N_hat:.0f}')
print()

Lincoln-Petersen estimate: N = N1 * N2 / Nb = 1139 * 1070 / 554 = 2200



## 3. Chapman correction (less biased for small samples)

$\hat{N}_C = \frac{(N_1 + 1)(N_2 + 1)}{N_b + 1} - 1$

In [12]:
N_chapman = ((N1 + 1) * (N2 + 1)) / (Nb + 1) - 1
print(f'Chapman estimate: N_c = {N_chapman:.0f}')
print()

Chapman estimate: N_c = 2199



## 4. Confidence interval (approximate 95% CI)

Variance (Chapman): $\text{Var}(\hat{N}_C) = \frac{(N_1+1)(N_2+1)(N_1-N_b)(N_2-N_b)}{(N_b+1)^2(N_b+2)}$

In [13]:
var_chapman = ((N1 + 1) * (N2 + 1) * (N1 - Nb) * (N2 - Nb)) / ((Nb + 1)**2 * (Nb + 2))
se = math.sqrt(var_chapman)
ci_lower = N_chapman - 1.96 * se
ci_upper = N_chapman + 1.96 * se

print(f'Standard error: {se:.1f}')
print(f'95% CI: [{ci_lower:.0f}, {ci_upper:.0f}]')

Standard error: 46.4
95% CI: [2108, 2290]


## 5. Summary

In [14]:
# Total unique individuals actually observed across both days
total_observed = len(N1_set | N2_set)
only_day1 = len(N1_set - N2_set)
only_day2 = len(N2_set - N1_set)

print('=' * 50)
print('POPULATION ESTIMATE SUMMARY')
print('=' * 50)
print(f'Day 1 individuals (N1):        {N1}')
print(f'Day 2 individuals (N2):        {N2}')
print(f'Recaptured (Nb):               {Nb}')
print(f'Seen only Day 1:               {only_day1}')
print(f'Seen only Day 2:               {only_day2}')
print(f'Total observed:                {total_observed}')
print(f'Recapture rate:                {Nb/N1:.1%} of Day 1 seen again on Day 2')
print('-' * 50)
print(f'Lincoln-Petersen estimate:     {N_hat:.0f}')
print(f'Chapman estimate:              {N_chapman:.0f}')
print(f'95% CI:                        [{ci_lower:.0f}, {ci_upper:.0f}]')
print('=' * 50)

POPULATION ESTIMATE SUMMARY
Day 1 individuals (N1):        1139
Day 2 individuals (N2):        1070
Recaptured (Nb):               554
Seen only Day 1:               585
Seen only Day 2:               516
Total observed:                1655
Recapture rate:                48.6% of Day 1 seen again on Day 2
--------------------------------------------------
Lincoln-Petersen estimate:     2200
Chapman estimate:              2199
95% CI:                        [2108, 2290]
